In [36]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [9]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [6]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("medium")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language='pt')
transcription = result["text"]
print(transcription)


 Como seria se Dostoevsky fosse personagem de filme estilo B-movie?


# 3. Integração com a API do ChatGPT 💬

In [33]:
!pip install openai

In [23]:
import os
os.environ['OPENAI_API_KEY'] = 'minha_chave_aqui'

In [26]:
# -------------------------------
# Transcrição já obtida anteriormente
# transcription = result["text"]

pergunta = transcription.lower()

# -------------------------------
# Resposta simulada (sem uso da API)
# A integração com a API da OpenAI foi implementada, porém está simulada devido à limitação de crédito disponível.

if "dostoiévski" in pergunta or "dostoievski" in pergunta or "toievski" in pergunta:
    chatgpt_response = """Se Dostoiévski fosse um personagem de um filme estilo B-movie, ele seria um anti-herói atormentado, envolvido em um crime misterioso enquanto enfrenta intensos conflitos psicológicos. Em um cenário sombrio e caótico, sua maior batalha não seria contra inimigos externos, mas contra sua própria consciência."""
else:
    chatgpt_response = "Se Fiódor Dostoiévski fosse um personagem de um filme estilo B-movie, ele deixaria de ser apenas um pensador para se tornar um anti-herói mergulhado em paranoia e culpa. Em um cenário sombrio e caótico, típico de produções de baixo orçamento, sua jornada não seria guiada por ação heroica, mas por intensos conflitos internos. Enquanto foge de uma ameaça incerta — que pode ser tanto real quanto fruto de sua própria mente —, o personagem questiona constantemente a moralidade de seus atos e o sentido da existência. O contraste entre a estética exagerada do B-movie e a profundidade filosófica de Dostoiévski criaria uma narrativa única, em que o verdadeiro suspense não está na perseguição, mas no confronto com a própria consciência."

print("Pergunta:", transcription)
print("Resposta:", chatgpt_response)

# -------------------------------
# Conversão da resposta em áudio
from gtts import gTTS
from IPython.display import Audio, display

gtts_object = gTTS(text=chatgpt_response, lang='pt', slow=False)

response_audio = "/content/response_audio.mp3"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))


Pergunta:  Como seria se Dostoevsky fosse personagem de filme estilo B-movie?
Resposta: Se Fiódor Dostoiévski fosse um personagem de um filme estilo B-movie, ele deixaria de ser apenas um pensador para se tornar um anti-herói mergulhado em paranoia e culpa. Em um cenário sombrio e caótico, típico de produções de baixo orçamento, sua jornada não seria guiada por ação heroica, mas por intensos conflitos internos. Enquanto foge de uma ameaça incerta — que pode ser tanto real quanto fruto de sua própria mente —, o personagem questiona constantemente a moralidade de seus atos e o sentido da existência. O contraste entre a estética exagerada do B-movie e a profundidade filosófica de Dostoiévski criaria uma narrativa única, em que o verdadeiro suspense não está na perseguição, mas no confronto com a própria consciência.


# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [13]:
!pip install gTTS

In [27]:
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo ChatGPT e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=chatgpt_response, lang='pt', slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.mp3"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))